In [1]:
from pathlib import Path

import pandas as pd
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score

In [2]:
EEG_DIR = Path(r"C:\Users\oieru\OneDrive\Escritorio\MASTER\00-TFM\00-TFM_FINAL\DataBase\Limpio2\Prepared\02_results_eeg_narrative_paper_folds_unimodal")
TS_DIR = Path(r"C:\Users\oieru\OneDrive\Escritorio\MASTER\00-TFM\00-TFM_FINAL\DataBase\Limpio2\Prepared\03_results_text_speech_xgboost_paper")

OUT_DIR = Path(r"C:\Users\oieru\OneDrive\Escritorio\MASTER\00-TFM\00-TFM_FINAL\DataBase\Limpio2\Prepared\08_results_late_fusion_text_speech_eeg_narrative")
OUT_DIR.mkdir(parents=True, exist_ok=True)

EEG_PRED_PATH = EEG_DIR / "eeg_subject_predictions.csv"
TS_PRED_PATH = TS_DIR / "text_speech_conversation_predictions.csv"

In [3]:
# EEG narrative-wise: una predicción por sujeto.
eeg = pd.read_csv(EEG_PRED_PATH)
eeg = eeg.rename(columns={"y_prob": "prob_eeg", "y_pred": "pred_eeg"})
eeg = eeg[["subject_id", "label", "outer_fold", "prob_eeg", "pred_eeg"]]

# Text+Speech: varias conversaciones por sujeto. Se agregan probabilidades por sujeto.
ts_conv = pd.read_csv(TS_PRED_PATH)
ts = (
    ts_conv
    .groupby(["subject_id", "label", "outer_fold"], as_index=False)["y_prob"]
    .mean()
    .rename(columns={"y_prob": "prob_ts"})
)
ts["pred_ts"] = (ts["prob_ts"] >= 0.5).astype(int)

print("Sujetos EEG:", eeg["subject_id"].nunique())
print("Sujetos Text+Speech:", ts["subject_id"].nunique())

Sujetos EEG: 94
Sujetos Text+Speech: 101


In [4]:
fusion = ts.merge(
    eeg[["subject_id", "label", "outer_fold", "prob_eeg"]],
    on=["subject_id", "label", "outer_fold"],
    how="inner",
)

fusion["prob_fusion"] = (fusion["prob_ts"] + fusion["prob_eeg"]) / 2
fusion["pred_fusion"] = (fusion["prob_fusion"] >= 0.5).astype(int)

print("Sujetos fusionados:", fusion["subject_id"].nunique())
display(fusion.head())

Sujetos fusionados: 94


,subject_id,label,outer_fold,prob_ts,pred_ts,prob_eeg,prob_fusion,pred_fusion
0,USER_01_CB,0,1,0.069472,0,0.270659,0.170066,0
1,USER_01_CB2,0,3,0.262455,0,0.357588,0.310021,0
2,USER_02_CB,0,5,0.134940,0,0.412303,0.273622,0
3,USER_02_CB2,0,3,0.192172,0,0.265462,0.228817,0
4,USER_03_CB,0,2,0.377053,0,0.407551,0.392302,0


Calcular métricas por outer fold


In [5]:
rows = []

for fold, fold_df in fusion.groupby("outer_fold"):
    y_true = fold_df["label"]
    y_pred = fold_df["pred_fusion"]

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    rows.append({
        "outer_fold": fold,
        "n_subjects": len(fold_df),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "Acc": accuracy_score(y_true, y_pred),
        "TPR": tp / (tp + fn),
        "TNR": tn / (tn + fp),
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp,
    })

metrics_by_fold = pd.DataFrame(rows).sort_values("outer_fold")
display(metrics_by_fold.round(3))

,outer_fold,n_subjects,F1,Acc,TPR,TNR,TN,FP,FN,TP
0,1,20,0.625,0.700,0.556,0.818,9,2,4,5
1,2,18,0.625,0.667,0.625,0.700,7,3,3,5
2,3,18,0.833,0.889,0.714,1.000,11,0,2,5
3,4,19,0.833,0.895,0.714,1.000,12,0,2,5
4,5,19,0.667,0.789,0.500,1.000,11,0,4,4


In [6]:
summary = pd.DataFrame({
    "mean": metrics_by_fold[["F1", "Acc", "TPR", "TNR"]].mean(),
    "std": metrics_by_fold[["F1", "Acc", "TPR", "TNR"]].std(),
}).round(3)

display(summary)

,mean,std
F1,0.717,0.108
Acc,0.788,0.105
TPR,0.622,0.095
TNR,0.904,0.138


In [7]:
fusion.to_csv(OUT_DIR / "late_fusion_subject_predictions.csv", index=False)
metrics_by_fold.to_csv(OUT_DIR / "late_fusion_metrics_by_fold.csv", index=False)
summary.to_csv(OUT_DIR / "late_fusion_summary.csv")

print("Resultados guardados en:")
print(OUT_DIR)

Resultados guardados en:
C:\Users\oieru\OneDrive\Escritorio\MASTER\00-TFM\00-TFM_FINAL\DataBase\Limpio2\Prepared\08_results_late_fusion_text_speech_eeg_narrative
